# Kaggle Worker · SANA Sprint 1.6B

**目标：Kaggle 2×T4，作为 `Kaggle Inference Hub` 的 `sana-sprint-1.6b` Worker。**

```text
Local Hub / Cloudflare Tunnel
        ↓ model-routed long polling
Kaggle SANA Worker
        ├── GPU0 → pipe0
        └── GPU1 → pipe1
        ↓ WebP + AES-GCM
Local Hub / Gallery
```


## 0. 可选：彻底清 GPU

如果当前 Kernel 之前加载过其它大模型，直接结束当前 Kernel 最干净。正常运行 Worker 时不要执行。


In [ ]:
# import os
# os._exit(0)


## 1. 下载 SANA Sprint 1.6B


In [2]:
import time
import os
import kagglehub

handle = "kawchar85/sana_sprint_1.6b_1024px/transformers/default/1"

start = time.perf_counter()

path = kagglehub.model_download(handle)

elapsed = time.perf_counter() - start

print(f"耗时：{elapsed:.2f} 秒")
print(f"路径：{path}")
print(f"路径存在：{os.path.exists(path)}")


耗时：1.27 秒
路径：/kaggle/input/models/kawchar85/sana_sprint_1.6b_1024px/transformers/default/1
路径存在：True


## 2. 双卡加载模型


In [ ]:
import torch
from diffusers import SanaSprintPipeline

assert torch.cuda.device_count()>=2,f"需要2张GPU，当前只有{torch.cuda.device_count()}张"

def load_pipe(i):
    d=f"cuda:{i}"; p=SanaSprintPipeline.from_pretrained(path,torch_dtype=torch.bfloat16,local_files_only=True); p.to(d); p.vae.to(device=d,dtype=torch.float32); print(f"GPU{i} {torch.cuda.get_device_name(i)} | {torch.cuda.memory_allocated(i)/1024**3:.2f}GiB"); return p

pipe0,pipe1=load_pipe(0),load_pipe(1)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

## 3. 启动 Hub Worker

只领取 `sana-sprint-1.6b` 队列；启动时注册 Worker，每 10 秒发送心跳。上传失败最多重试 3 次，生成失败会通知本地 Hub 按尝试次数决定是否重新入队。

只需要按实际情况修改 `BASE` 与 `PASSWORD`。


In [ ]:
# !pip install -q aiohttp cryptography requests

import os,time,queue,threading,asyncio,io,hashlib,torch,aiohttp,requests,uuid
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

BASE = "https://ranran-sana.202820.xyz"
MODEL = "sana-sprint-1.6b"
PASSWORD = "wangran"
WORKER_ID = f"sana-{uuid.uuid4().hex[:8]}"
TASK_URL = f"{BASE}/task/next?model={MODEL}&worker_id={WORKER_ID}"
UPLOAD_URL = f"{BASE}/upload"
REGISTER_URL = f"{BASE}/worker/register"
HEARTBEAT_URL = f"{BASE}/worker/heartbeat"
FAIL_URL = f"{BASE}/task/fail"

KEY=hashlib.sha256(PASSWORD.encode()).digest(); IO_WORKERS=4
tasks=queue.Queue(maxsize=32); uploads=queue.Queue(maxsize=64); STOP=threading.Event()


def encode(image):
    b=io.BytesIO(); image.save(b,"WEBP",quality=90,method=4); return b.getvalue()


def encrypt(data):
    nonce=os.urandom(12); return nonce+AESGCM(KEY).encrypt(nonce,data,None)


async def upload_image(x,session):
    try:
        data=await asyncio.to_thread(encode,x["image"]); data=await asyncio.to_thread(encrypt,data)
        last=None
        for attempt in range(3):
            form=aiohttp.FormData(); form.add_field("file",data,filename=f'{x["id"]:04d}.bin',content_type="application/octet-stream")
            for k in ("id","model","worker_id","gpu","seed","prompt","seconds","steps"): form.add_field(k,str(x[k]))
            try:
                async with session.post(UPLOAD_URL,data=form) as r:
                    if r.status>=400: raise RuntimeError(f"HTTP {r.status}: {await r.text()}")
                last=None; break
            except Exception as e:
                last=e
                if attempt<2: await asyncio.sleep(2**attempt)
        if last: raise last
        print(f'↑ #{x["id"]:03d} | GPU{x["gpu"]} → PC')
    except Exception as e: print(f'! Upload #{x.get("id")}: {e}')
    finally: x.pop("image",None); uploads.task_done()


async def upload_worker(session):
    while True:
        x=await asyncio.to_thread(uploads.get)
        if x is None: uploads.task_done(); break
        await upload_image(x,session)


async def task_poller(session):
    while not STOP.is_set():
        try:
            async with session.get(TASK_URL) as r:
                if r.status==204: continue
                if r.status>=400: raise RuntimeError(f"HTTP {r.status}: {await r.text()}")
                x=await r.json(); await asyncio.to_thread(tasks.put,x); print(f'↓ #{x["id"]:03d} | {x["prompt"][:70]}')
        except Exception as e:
            if not STOP.is_set(): print(f"! Poller: {e}"); await asyncio.sleep(2)


async def heartbeat(session):
    while not STOP.is_set():
        try:
            async with session.post(HEARTBEAT_URL,json={"worker_id":WORKER_ID,"local_queue":tasks.qsize(),"upload_queue":uploads.qsize()}) as r:
                if r.status==404:
                    await register_worker(session)
        except Exception as e: print(f"! Heartbeat: {e}")
        await asyncio.sleep(10)


async def register_worker(session):
    payload={"worker_id":WORKER_ID,"model":MODEL,"gpus":[torch.cuda.get_device_name(0),torch.cuda.get_device_name(1)],"runtime":"diffusers","concurrency":2}
    async with session.post(REGISTER_URL,json=payload) as r:
        if r.status>=400: raise RuntimeError(f"register HTTP {r.status}: {await r.text()}")
    print(f"✓ registered {WORKER_ID} -> {MODEL}")


async def network_main():
    timeout=aiohttp.ClientTimeout(total=40)
    async with aiohttp.ClientSession(headers={"Authorization":f"Bearer {PASSWORD}"},timeout=timeout) as session:
        await register_worker(session)
        await asyncio.gather(task_poller(session),heartbeat(session),*(upload_worker(session) for _ in range(IO_WORKERS)))


def run_network(): asyncio.run(network_main())


def report_fail(task_id,error):
    try:
        requests.post(FAIL_URL,headers={"Authorization":f"Bearer {PASSWORD}"},json={"id":task_id,"error":str(error),"requeue":True},timeout=15)
    except Exception as e: print(f"! fail report #{task_id}: {e}")


def gpu_worker(i,p):
    d=f"cuda:{i}"; torch.cuda.set_device(i)
    while True:
        x=tasks.get()
        if x is None: tasks.task_done(); break
        start=time.perf_counter()
        try:
            g=torch.Generator(device=d).manual_seed(int(x["seed"]))
            with torch.inference_mode():
                image=p(prompt=x["prompt"],num_inference_steps=int(x["steps"]),guidance_scale=0.0,width=int(x["width"]),height=int(x["height"]),generator=g).images[0]
            sec=round(time.perf_counter()-start,3)
            uploads.put({"id":x["id"],"model":MODEL,"worker_id":WORKER_ID,"gpu":i,"seed":x["seed"],"prompt":x["prompt"],"seconds":sec,"steps":x["steps"],"image":image})
            print(f'✓ #{x["id"]:03d} | GPU{i} | {sec:.2f}s')
        except Exception as e:
            print(f'✗ #{x.get("id")} | GPU{i} | {e}'); report_fail(x["id"],e)
        finally: tasks.task_done()


network_thread=threading.Thread(target=run_network,daemon=True); network_thread.start()
gpu_workers=[threading.Thread(target=gpu_worker,args=(0,pipe0),daemon=True),threading.Thread(target=gpu_worker,args=(1,pipe1),daemon=True)]
for w in gpu_workers: w.start()
print(f"✓ SANA Worker {WORKER_ID} 已启动 | GPU0 + GPU1 | 等待 {MODEL} Prompt...")


## 4. 状态检查


In [ ]:
print("worker_id:", WORKER_ID)
print("model:", MODEL)
print("STOP:", STOP.is_set())
print("task queue:", tasks.qsize(), "/", tasks.maxsize)
print("upload queue:", uploads.qsize(), "/", uploads.maxsize)


## 5. 停止继续领取新任务

设置 `STOP` 后不再继续轮询。要彻底释放两个 Pipeline 的显存，建议直接重启 Kernel。


In [ ]:
STOP.set()
print("✓ STOP 已设置")
